# Step 4a — Closed Book: Answering from Memory

We have a vetted quiz about *very recent* news (Step 3). Now we bring in the
answering models and ask the tutorial's central question: **where does an
LLM's knowledge actually come from?** We test each model three ways, one
notebook per method:

| Method | Notebook | What the model gets | What it measures |
|---|---|---|---|
| `closed_book` | **this one** | question + options, nothing else | what's in the weights |
| `web_search` | [`04b_web_search.ipynb`](04b_web_search.ipynb) | same prompt, search tool ON | what retrieval adds |
| `debate` | [`04c_debate.ipynb`](04c_debate.ipynb) | 3 copies of the model argue over 3 rounds | what deliberation adds |

The questions were written from articles published *after* these models'
training data was collected — so closed-book answers must come from stale
weights or lucky guessing. This notebook builds the closed-book condition
from a raw SDK call up to the toolkit function, then digs into one knob
every API exposes but audits often ignore: **sampling temperature**.

## 1. What the answering model sees

Almost nothing. Unlike the judge (who got the full article), an answering model
gets the question and four lettered options — **never the article**. If we
leaked the article, every method would score 100% and measure only reading
comprehension:

In [1]:
from toolkit import prompts
from toolkit.utils import load_jsonl

selected = load_jsonl("../../data/questions/selected_questions.jsonl")
question = selected[0]

user_prompt = prompts.build_answer_user_prompt(
    question["question"], question["options"]
)
print(user_prompt)

QUESTION:
Which company conducted the AI detection review of Pope Leo XIV's collection of speeches and writings, Maps of Hope?

OPTIONS:
A. Breaking News Australia
B. Proudly Human
C. Australian Catholic University
D. The Vatican Publishing House



The system half sets the role and — crucially — forces a commitment: pick
exactly one letter, always, plus a confidence between 0 and 1. Refusals and
"it depends" answers would be ungradable:

In [2]:
print(prompts.ANSWER_SYSTEM_PROMPT)

You are an expert news-quiz contestant. Each question was written from a
recently published news article (within the last few weeks). You are NOT
given the article — answer from what you know or can find.

Rules:
- Pick the single best option: exactly one of A, B, C, or D.
- Always commit to one letter, even if you are unsure.
- Give 1-2 sentences of reasoning and a confidence between 0 and 1.



## 2. One raw closed-book call

This is the same two-message + Pydantic-schema pattern you built in Step 2
— only the schema changes. The answering contract is three fields:

```python
class Answer(BaseModel):
    answer_letter: Literal["A", "B", "C", "D"]
    confidence: float          # 0-1, self-reported
    reasoning: str             # 1-2 sentences
```

Straight to the OpenAI SDK: system prompt as the `developer` message, the
question as the `user` message, `text_format=Answer`:

In [3]:
from openai import OpenAI

from toolkit.answers import Answer
from toolkit.providers import PROVIDER_ENV, load_api_key

client = OpenAI(api_key=load_api_key(PROVIDER_ENV["openai"]))

response = client.responses.parse(
    model="gpt-5.4-mini-2026-03-17",
    input=[
        {"role": "developer", "content": prompts.ANSWER_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ],
    text_format=Answer,
)

answer = response.output_parsed      # <- a validated Answer object
answer

Answer(answer_letter='B', confidence=0.91, reasoning="The AI detection review of Pope Leo XIV’s collection 'Maps of Hope' was conducted by Proudly Human, a firm focused on AI and human-authorship verification. The other options are publishers or institutions, not the reviewing company.")

That's the whole method. Grading is one comparison against the answer key
that Step 3's judges vetted:

In [4]:
is_correct = answer.answer_letter == question["correct_letter"]

print(question["question"], "\n")
for letter, option in zip("ABCD", question["options"]):
    mark = "*" if letter == question["correct_letter"] else " "
    print(f"  {mark}{letter}. {option}")
print(f"\nanswered {answer.answer_letter} "
      f"(confidence {answer.confidence:.2f}) -> "
      f"{'CORRECT' if is_correct else 'WRONG'}")
print("WHY:", answer.reasoning)

Which company conducted the AI detection review of Pope Leo XIV's collection of speeches and writings, Maps of Hope? 

   A. Breaking News Australia
  *B. Proudly Human
   C. Australian Catholic University
   D. The Vatican Publishing House

answered B (confidence 0.91) -> CORRECT
WHY: The AI detection review of Pope Leo XIV’s collection 'Maps of Hope' was conducted by Proudly Human, a firm focused on AI and human-authorship verification. The other options are publishers or institutions, not the reviewing company.


## 3. This is all in the toolkit

`answer_question()` wraps exactly what we just did by hand, plus the
bookkeeping a real experiment needs:

- it looks up the model's provider in `config.ANSWER_MODELS` and calls the
  matching adapter (`openai_provider.run_parsed` / `gemini_provider.run_parsed`
  — the same interchangeable-signature trick from Step 2, with tenacity
  retries built in);
- it builds the prompts from the question record, so you can't accidentally
  leak the article;
- it grades the answer and returns a flat, JSONL-ready **record** with ids,
  model, method, and a timestamp — the row format every later step reads:

In [5]:
from toolkit.answers import answer_question

closed = answer_question(
    question, model="gpt-5.4-mini-2026-03-17", method="closed_book"
)

{k: v for k, v in closed.items() if k not in ("raw", "reasoning")}

{'id': 'closed_book__gpt-5.4-mini-2026-03-17__gemini__world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence__q0',
 'question_id': 'gemini__world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence__q0',
 'article_id': 'world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence',
 'method': 'closed_book',
 'model': 'gpt-5.4-mini-2026-03-17',
 'provider': 'openai',
 'answer_letter': 'B',
 'correct_letter': 'B',
 'is_correct': True,
 'confidence': 0.93,
 'search_used': None,
 'debate': None,
 'answered_at': '2026-07-26T19:38:47.755016+00:00'}

## 4. Parameter deep-dive: temperature

A language model doesn't produce *the* next token — it produces a
probability distribution over every possible next token, and then
**samples** from it. Temperature rescales that distribution before
sampling:

- **Low temperature (→ 0)** sharpens the distribution: the most likely
  token wins almost every time, so repeated runs give near-identical
  answers.
- **High temperature** flattens it: lower-probability tokens get real
  chances, so runs diverge — sometimes creatively, sometimes into the
  wrong answer letter.

Why an *audit* should care: accuracy measured at one temperature is a
sample from a distribution, not a constant. If answers flip from run to
run, a single pass over 100 questions carries hidden variance — and a
result nobody can reproduce without knowing the setting.

Two provider wrinkles worth knowing. First: the GPT-5-series models used
in this tutorial are *reasoning* models, and OpenAI rejects non-default
`temperature` for them (only the default `1` is allowed — the API
returns a 400 otherwise). Gemini accepts the full 0.0–2.0 range, so we
switch to `gemini-3.1-flash-lite`. Second: temperature only matters when
the model is genuinely torn — on a question it answers at confidence
0.9+, the distribution is so peaked that even a high temperature rarely
flips the letter. So for this experiment we pick a question the model is
*less* sure about: a fiscal figure whose distractors are nearby numbers,
exactly the kind of option set where probability mass spreads out.

In the raw SDK, temperature is one more line in the
`GenerateContentConfig` (alongside Step 2's `system_instruction` +
`response_schema` pattern):

In [6]:
from google import genai
from google.genai import types

gclient = genai.Client(api_key=load_api_key(PROVIDER_ENV["gemini"]))

temp_question = selected[3]          # the harder, numbers question
print(temp_question["question"], temp_question["options"], "\n")

gresponse = gclient.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompts.build_answer_user_prompt(
        temp_question["question"], temp_question["options"]
    ),
    config=types.GenerateContentConfig(
        system_instruction=prompts.ANSWER_SYSTEM_PROMPT,
        response_mime_type="application/json",
        response_schema=Answer,
        temperature=2.0,             # <- the knob (0.0-2.0)
    ),
)

gresponse.parsed

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


According to the article, how much does the Office for Budget Responsibility estimate the Bank of England's sale of bonds added to the budget deficit in the current fiscal year? ['£2 billion', '£6 billion', '£10 billion', '£15 billion'] 



Answer(answer_letter='C', confidence=0.9, reasoning="The Office for Budget Responsibility reported that the Bank of England's quantitative tightening process has increased the deficit by £10 billion for the current fiscal year due to rising interest costs on central bank reserves.")

The toolkit threads the same knob through as an optional `temperature`
kwarg on `answer_question()` — leave it unset (the default `None`) and
the provider's own default applies, which is what every other call in
the tutorial uses.

Now the actual experiment: the same question, five times at `0.0` and
five times at `2.0`:

In [7]:
import pandas as pd

RUNS = 5
rows = []
for temperature in (0.0, 2.0):
    for run in range(RUNS):
        r = answer_question(
            temp_question,
            model="gemini-3.1-flash-lite",
            method="closed_book",
            temperature=temperature,
        )
        rows.append(
            {
                "temperature": temperature,
                "run": run,
                "answer_letter": r["answer_letter"],
                "confidence": r["confidence"],
                "is_correct": r["is_correct"],
            }
        )

runs_df = pd.DataFrame(rows)
runs_df

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


,temperature,run,answer_letter,confidence,is_correct
0,0.0,0,C,0.90,False
1,0.0,1,C,0.90,False
2,0.0,2,C,0.90,False
3,0.0,3,C,0.90,False
4,0.0,4,C,0.90,False
5,2.0,0,D,0.90,False
6,2.0,1,D,0.95,False
7,2.0,2,D,0.90,False
8,2.0,3,B,0.90,True
9,2.0,4,D,0.90,False


In [8]:
summary = runs_df.groupby("temperature").agg(
    distinct_letters=("answer_letter", "nunique"),
    letters=("answer_letter", lambda s: "".join(s)),
    accuracy=("is_correct", "mean"),
    mean_confidence=("confidence", "mean"),
)
summary

,distinct_letters,letters,accuracy,mean_confidence
temperature,,,,
0.0,1,CCCCC,0.0,0.90
2.0,2,DDDBD,0.2,0.91


What to look for in the table:

- **Letter stability.** At `0.0` the five letters are usually (not
  always!) identical; at `2.0` they are far more likely to differ.
  Temperature 0 shrinks randomness but does not guarantee determinism —
  provider-side factors (batching, hardware) still wiggle the logits, so
  even the "deterministic" row can flip occasionally.
- **Stability is not accuracy.** Temperature 0 makes the model
  *consistent*, not *right* — it can pick the same wrong letter five
  times in a row. Low temperature buys reproducibility; the knowledge in
  the weights is whatever it is.
- **Accuracy as a random variable.** When letters flip, `accuracy` in
  this tiny 5-run sample moves in steps of 0.2. The same effect —
  smaller but real — is buried inside any single-pass benchmark run.
- **Confidence doesn't track temperature.** The self-reported confidence
  is part of the *sampled text*, not a readout of the distribution, so
  don't expect it to drop just because sampling got noisier.

The takeaway for the full experiment: **fix the setting and report it.**
The tutorial's pipeline leaves `temperature` unset everywhere, i.e. each
provider's default — that choice is part of the method, and now it's a
documented one.

## 5. The full experiment, from the command line

Answering 100 questions at scale is script work. One run = one method ×
one model = one JSONL file, with crash-safe append + resume. The script
runs at provider-default temperature:

```bash
# 04-1: closed book, all six models (600 calls)
for M in gpt-5.4-mini-2026-03-17 gpt-5.5-2026-04-23 gpt-5.6-luna \
         gpt-5.6-terra gemini-3.1-flash-lite gemini-3.5-flash; do
  uv run python scripts/04-1_generate_answers.py \
      --model $M --method closed_book --parallel
done
```

## 6. The map

| This notebook | Where it lives |
|---|---|
| §1 answering prompts | `toolkit.prompts.ANSWER_SYSTEM_PROMPT`, `build_answer_user_prompt()` |
| §2 raw call + schema | `toolkit.answers.Answer`; wrapped by `toolkit.providers.openai_provider.run_parsed()` |
| §3 one graded record | `toolkit.answers.answer_question()`, `to_answer_record()` |
| §4 temperature | the `temperature` kwarg on `answer_question()`, threaded through both provider adapters' `_call` |
| §5 at scale | `scripts/04-1_generate_answers.py`; `toolkit.answers.answer_questions()` |

---

### Next up 🔎

Same prompt, one flag flipped: [`04b_web_search.ipynb`](04b_web_search.ipynb)
hands the model a live search tool — and then tells it *where it is allowed
to look*.